# Ships and Ports in Canada
**Oritse-Tsegbemi Eyito | OSS4008A**

**Meeting demo:** Run all, then click **Load published demo files** above the map. Select one of the archived predictions under **Saved run** to show it without Earth Engine authorization.

This Colab notebook opens the national facility map and runs ship detection on selected Sentinel-2 scenes. Use the existing trained checkpoint for inference. Run from the top to initialize the dashboard.

The notebook imports Python modules from an existing course clone. The embedded copies support standalone Colab sessions.


In [ ]:
%pip -q install earthengine-api geemap ipywidgets rasterio pyproj pycocotools geopandas pyogrio


In [ ]:
import json
import sys
from pathlib import Path
from google.colab import output
from IPython.display import display

output.enable_custom_widget_manager()
repo = Path('/content/OSS4008A---Introduction-to-Remote-Sensing-and-Applications')
workspace = repo if (repo/'.git').is_dir() else Path('/content/phase3_work')
PROJECT_DIR = workspace/'students/oritse-tsegbemi_eyito'
for folder in ('code', 'data', 'results'):
    (PROJECT_DIR/folder).mkdir(parents=True, exist_ok=True)
if str(PROJECT_DIR/'code') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR/'code'))
print('Project folder:', PROJECT_DIR)


In [ ]:
module_path = PROJECT_DIR / 'code' / 'ship_backend.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Reproducible Sentinel-2 vessel detection backend. Run in Google Colab."""\nfrom pathlib import Path\nfrom collections import OrderedDict\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport torch\nimport torchvision\nfrom PIL import Image, ImageDraw\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\nfrom torchvision.models.detection import FasterRCNN\nfrom torchvision.models.detection.backbone_utils import resnet_fpn_backbone\nfrom torchvision.models import ResNet18_Weights\nfrom torchvision.models.detection.faster_rcnn import TwoMLPHead\n\nDATA_URL = \'https://zenodo.org/records/15571607/files/SEN2MS_Vessel_BBoxes.zip?download=1\'\nDATA_MD5 = \'830a5e741425c736b3971055ff2e9a9f\'\nDATA_DOI = \'https://doi.org/10.5281/zenodo.15571607\'\nDATA_LICENSE = \'CC-BY-4.0\'\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef checksum(path, algorithm=\'md5\'):\n    digest = hashlib.new(algorithm)\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef download_dataset(project):\n    """Download data only; never execute code from the archive."""\n    raw = Path(project) / \'data/raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    destination = raw / \'SEN2MS_Vessel_BBoxes.zip\'\n    if not destination.exists() or checksum(destination) != DATA_MD5:\n        temporary = destination.with_suffix(\'.partial\')\n        with requests.get(DATA_URL, stream=True, timeout=120) as response:\n            response.raise_for_status()\n            with temporary.open(\'wb\') as handle:\n                for chunk in response.iter_content(4 * 1024 * 1024):\n                    handle.write(chunk)\n        if checksum(temporary) != DATA_MD5:\n            raise ValueError(\'Dataset checksum differs from the published archive.\')\n        temporary.replace(destination)\n    print(\'Dataset archive verified:\', destination.stat().st_size, \'bytes\')\n    return destination\n\n\ndef prepare_dataset(project, archive_path, seed=42):\n    """Convert original labels to Labelme JSON and split by MGRS tile.\n\n    Image identity includes country AND source category: the original archive\n    reuses numeric image names in vessel/non-vessel directories.\n    """\n    project = Path(project)\n    image_dir, annotation_dir = project / \'data/images\', project / \'data/annotations\'\n    image_dir.mkdir(parents=True, exist_ok=True)\n    annotation_dir.mkdir(parents=True, exist_ok=True)\n    rows, seen, exclusions = [], set(), {\'cloud\': 0, \'duplicate_rgb\': 0}\n    with zipfile.ZipFile(archive_path) as archive:\n        members = set(archive.namelist())\n        for name in sorted(n for n in members if n.endswith(\'.csv\')):\n            table = pd.read_csv(io.BytesIO(archive.read(name)))\n            parent = name.rsplit(\'/\', 1)[0]\n            country, category = parent.split(\'/\')[-2:]\n            for _, source in table.iterrows():\n                if int(source[\'Clouds\']) > 1:\n                    exclusions[\'cloud\'] += 1\n                    continue\n                stem = Path(source[\'Image_name\']).stem\n                member = f\'{parent}/MS/{stem}.npy\'\n                if member not in members:\n                    raise FileNotFoundError(member)\n                array = np.load(io.BytesIO(archive.read(member)), allow_pickle=False)\n                if array.shape != (13, 128, 128) or not np.isfinite(array).all():\n                    raise ValueError(f\'Unexpected source array: {member}\')\n                # Source documentation indexes B04/B03/B02 at 3/2/1 and stretches\n                # each band to 0..255. Do not divide these stored values by 10000.\n                rgb = np.rint(np.clip(array[[3, 2, 1]], 0, 255)).astype(np.uint8)\n                digest = hashlib.sha256(rgb.tobytes()).hexdigest()\n                if digest in seen:\n                    exclusions[\'duplicate_rgb\'] += 1\n                    continue\n                seen.add(digest)\n                label_member = f\'{parent}/BBoxes/{stem}.txt\'\n                if category == \'vessel\' and label_member not in members:\n                    raise FileNotFoundError(f\'Missing vessel labels: {label_member}\')\n                boxes = []\n                if label_member in members:\n                    for line in archive.read(label_member).decode().splitlines():\n                        if not line.strip():\n                            continue\n                        label, cx, cy, width, height = map(float, line.split())\n                        if label != 0 or not (0 < width <= 1 and 0 < height <= 1):\n                            raise ValueError(f\'Invalid YOLO label: {label_member}\')\n                        box = np.array([cx-width/2, cy-height/2, cx+width/2, cy+height/2]) * 128\n                        box = np.clip(box, 0, 128).tolist()\n                        if box[2] <= box[0] or box[3] <= box[1]:\n                            raise ValueError(\'Degenerate bounding box.\')\n                        boxes.append(box)\n                tile_match = re.search(r\'_T(\\d{2}[A-Z]{3})_\', str(source[\'Picture_title\']))\n                if not tile_match:\n                    raise ValueError(\'Source MGRS tile is missing.\')\n                identity = f\'{country}_{category}_{stem}\'\n                image_path = image_dir / f\'{identity}.png\'\n                Image.fromarray(rgb.transpose(1, 2, 0)).save(image_path)\n                annotation = {\n                    \'version\': \'5.0.1\', \'flags\': {},\n                    \'shapes\': [{\'label\': \'ship\', \'points\': [[b[0], b[1]], [b[2], b[3]]],\n                                \'group_id\': None, \'shape_type\': \'rectangle\', \'flags\': {}} for b in boxes],\n                    \'imagePath\': f\'../images/{identity}.png\', \'imageData\': None,\n                    \'imageHeight\': 128, \'imageWidth\': 128,\n                }\n                annotation_path = annotation_dir / f\'{identity}.json\'\n                write_json(annotation_path, annotation)\n                rows.append({\'image_id\': identity, \'country\': country,\n                             \'source_tile\': tile_match.group(1), \'source_scene\': source[\'Picture_title\'],\n                             \'image_path\': str(image_path.relative_to(project)),\n                             \'annotation_path\': str(annotation_path.relative_to(project)),\n                             \'boxes\': len(boxes), \'rgb_sha256\': digest})\n    manifest = pd.DataFrame(rows)\n    # Fixed geographic groups, chosen without looking at model performance.\n    # Stratify tile lists by whether any ship labels are available in that tile.\n    tile_has_ship = manifest.groupby(\'source_tile\')[\'boxes\'].sum().gt(0)\n    assignment = {}\n    rng = random.Random(seed)\n    for positive in [True, False]:\n        tiles = sorted(tile_has_ship[tile_has_ship == positive].index)\n        rng.shuffle(tiles)\n        count = max(1, round(len(tiles) * 0.2)) if len(tiles) >= 3 else 0\n        for index, tile in enumerate(tiles):\n            assignment[tile] = \'test\' if index < count else \'val\' if index < 2*count else \'train\'\n    manifest[\'split\'] = manifest.source_tile.map(assignment)\n    for split in [\'train\', \'val\', \'test\']:\n        subset = manifest[manifest.split == split]\n        if subset.empty or subset.boxes.sum() == 0:\n            raise ValueError(f\'{split} has no usable ship examples.\')\n    assert manifest.groupby(\'source_tile\').split.nunique().max() == 1\n    assert manifest.rgb_sha256.is_unique\n    manifest.to_csv(project / \'data/dataset_manifest.csv\', index=False)\n    summary = {\'source\': DATA_DOI, \'license\': DATA_LICENSE, \'archive_md5\': DATA_MD5,\n               \'source_processing\': \'Sentinel-2 L1C, source per-band percentile stretch to 0..255\',\n               \'input_bands\': [\'B4\', \'B3\', \'B2\'], \'seed\': seed,\n               \'split_method\': \'MGRS tile groups, 60/20/20 approximate; exact RGB duplicates removed\',\n               \'exclusions\': exclusions,\n               \'splits\': {s: {\'images\': int((manifest.split == s).sum()),\n                              \'boxes\': int(manifest.loc[manifest.split == s, \'boxes\'].sum()),\n                              \'tiles\': sorted(manifest.loc[manifest.split == s, \'source_tile\'].unique())}\n                          for s in [\'train\', \'val\', \'test\']}}\n    write_json(project / \'results/dataset_audit.json\', summary)\n    print(json.dumps(summary, indent=2))\n    return manifest\n\n\nclass VesselDataset(Dataset):\n    def __init__(self, project, table, augment=False):\n        self.project, self.rows, self.augment = Path(project), table.to_dict(\'records\'), augment\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        image = np.array(Image.open(self.project / row[\'image_path\']).convert(\'RGB\'))\n        image = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255\n        annotation = json.loads((self.project / row[\'annotation_path\']).read_text())\n        boxes = torch.tensor([s[\'points\'][0]+s[\'points\'][1] for s in annotation[\'shapes\']],\n                             dtype=torch.float32).reshape(-1, 4)\n        if self.augment and random.random() < 0.5:\n            image = image.flip(-1)\n            boxes[:, [0, 2]] = 128 - boxes[:, [2, 0]]\n        target = {\'boxes\': boxes, \'labels\': torch.ones(len(boxes), dtype=torch.int64),\n                  \'image_id\': torch.tensor(index),\n                  \'area\': (boxes[:, 2]-boxes[:, 0])*(boxes[:, 3]-boxes[:, 1]),\n                  \'iscrowd\': torch.zeros(len(boxes), dtype=torch.int64)}\n        return image, target\n\n\ndef collate(batch):\n    return tuple(zip(*batch))\n\n\ndef build_mobilenet_detector(pretrained=True):\n    """Transfer a COCO MobileNet detector to one vessel class and small boxes."""\n    model = fasterrcnn_mobilenet_v3_large_fpn(\n        weights=\'DEFAULT\' if pretrained else None, weights_backbone=None,\n        min_size=256, max_size=256, box_score_thresh=0.01,\n        rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=300,\n        rpn_pre_nms_top_n_test=600, rpn_post_nms_top_n_test=200,\n        box_detections_per_img=100,\n    )\n    original = model.roi_heads.box_predictor\n    predictor = FastRCNNPredictor(original.cls_score.in_features, 2)\n    if pretrained:\n        # COCO category 9 is boat. Preserve its learned classifier/regressor as\n        # the initial ship class instead of starting the output layer randomly.\n        with torch.no_grad():\n            predictor.cls_score.weight.copy_(original.cls_score.weight[[0, 9]])\n            predictor.cls_score.bias.copy_(original.cls_score.bias[[0, 9]])\n            predictor.bbox_pred.weight.copy_(torch.cat([original.bbox_pred.weight[:4], original.bbox_pred.weight[36:40]]))\n            predictor.bbox_pred.bias.copy_(torch.cat([original.bbox_pred.bias[:4], original.bbox_pred.bias[36:40]]))\n    model.roi_heads.box_predictor = predictor\n    # Preserve 15 anchors/location so the pretrained RPN output dimensions match.\n    model.rpn.anchor_generator = AnchorGenerator(((4, 8, 16, 32, 64),)*3, ((0.5, 1., 2.),)*3)\n    return model\n\n\ndef build_detector(pretrained=True):\n    """ImageNet-pretrained ResNet18 with stride-4 FPN features for small ships.\n\n    A smaller 512-unit ROI head keeps the full half-precision model below 50 MB.\n    This is a transfer-learning detector; the RPN and detection heads are learned\n    from the vessel annotations, while the backbone starts from ImageNet weights.\n    """\n    backbone = resnet_fpn_backbone(\'resnet18\',\n                                  weights=ResNet18_Weights.DEFAULT if pretrained else None,\n                                  trainable_layers=3)\n    anchors = AnchorGenerator(((4,), (8,), (16,), (32,), (64,)), ((0.5,1.,2.),)*5)\n    head = TwoMLPHead(256*7*7, 512)\n    predictor = FastRCNNPredictor(512, 2)\n    return FasterRCNN(backbone, num_classes=None, min_size=256, max_size=256,\n                      rpn_anchor_generator=anchors, box_head=head, box_predictor=predictor,\n                      box_score_thresh=0.01, box_detections_per_img=100,\n                      rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=300,\n                      rpn_pre_nms_top_n_test=600, rpn_post_nms_top_n_test=200)\n\n\ndef evaluate_detector(model, dataset, device, score_threshold=0.3):\n    """COCO AP plus greedy one-to-one precision/recall at IoU 0.50."""\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    model.eval()\n    annotations, detections, images = [], [], []\n    tp = fp = fn = 0\n    started = time.perf_counter()\n    with torch.inference_mode():\n        for index in range(len(dataset)):\n            image, target = dataset[index]\n            prediction = model([image.to(device)])[0]\n            boxes, scores = prediction[\'boxes\'].cpu(), prediction[\'scores\'].cpu()\n            images.append({\'id\': index, \'width\': 128, \'height\': 128})\n            for box in target[\'boxes\'].tolist():\n                x1, y1, x2, y2 = box\n                annotations.append({\'id\': len(annotations)+1, \'image_id\': index,\n                                    \'category_id\': 1, \'bbox\': [x1, y1, x2-x1, y2-y1],\n                                    \'area\': (x2-x1)*(y2-y1), \'iscrowd\': 0})\n            for box, score in zip(boxes.tolist(), scores.tolist()):\n                x1, y1, x2, y2 = box\n                detections.append({\'image_id\': index, \'category_id\': 1,\n                                   \'bbox\': [x1, y1, x2-x1, y2-y1], \'score\': score})\n            selected = boxes[scores >= score_threshold]\n            gt = target[\'boxes\']\n            matched = set()\n            overlaps = torchvision.ops.box_iou(selected, gt)\n            for overlap in overlaps:\n                options = [(float(v), j) for j, v in enumerate(overlap) if j not in matched]\n                best = max(options, default=(0., -1))\n                if best[0] >= 0.5:\n                    tp += 1\n                    matched.add(best[1])\n                else:\n                    fp += 1\n            fn += len(gt)-len(matched)\n    ground_truth = COCO()\n    ground_truth.dataset = {\'images\': images, \'annotations\': annotations,\n                            \'categories\': [{\'id\': 1, \'name\': \'ship\'}], \'info\': {}}\n    ground_truth.createIndex()\n    if detections:\n        result = ground_truth.loadRes(detections)\n        evaluation = COCOeval(ground_truth, result, \'bbox\')\n        evaluation.evaluate()\n        evaluation.accumulate()\n        evaluation.summarize()\n        ap, ap50 = map(float, evaluation.stats[:2])\n    else:\n        ap = ap50 = 0.\n    return {\'images\': len(dataset), \'ground_truth_boxes\': len(annotations),\n            \'ap_50_95\': ap, \'ap_50\': ap50,\n            \'precision_at_threshold\': tp/max(tp+fp, 1), \'recall_at_threshold\': tp/max(tp+fn, 1),\n            \'score_threshold\': score_threshold, \'iou_threshold\': 0.5,\n            \'true_positives\': tp, \'false_positives\': fp, \'false_negatives\': fn,\n            \'seconds\': time.perf_counter()-started}\n\n\ndef train_detector(project, manifest, epochs=20, seed=42, architecture=\'resnet18\'):\n    project = Path(project)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(2)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if architecture not in [\'resnet18\', \'mobilenet\']:\n        raise ValueError(\'Unknown detector architecture.\')\n    model = (build_detector(pretrained=True) if architecture==\'resnet18\'\n             else build_mobilenet_detector(pretrained=True)).to(device)\n    training = manifest[manifest.split == \'train\']\n    # Balance the training sample only. Validation/test retain all eligible data.\n    positive, negative = training[training.boxes > 0], training[training.boxes == 0]\n    training = pd.concat([positive, negative.sample(min(len(negative), len(positive)), random_state=seed)])\n    train_set = VesselDataset(project, training, augment=True)\n    val_set = VesselDataset(project, manifest[manifest.split == \'val\'])\n    test_set = VesselDataset(project, manifest[manifest.split == \'test\'])\n    loader = DataLoader(train_set, batch_size=8 if device.type == \'cuda\' else 2,\n                        shuffle=True, collate_fn=collate, num_workers=0)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],\n                                  lr=0.0001 if architecture==\'resnet18\' else 0.00015, weight_decay=0.0001)\n    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.3)\n    best_ap, history, best_state = -1., [], None\n    for epoch in range(epochs):\n        model.train()\n        losses, started = [], time.perf_counter()\n        for images, targets in loader:\n            images = [image.to(device) for image in images]\n            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]\n            loss = sum(model(images, targets).values())\n            if not torch.isfinite(loss):\n                raise ValueError(\'Non-finite training loss.\')\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        validation = evaluate_detector(model, val_set, device)\n        scheduler.step()\n        record = {\'epoch\': epoch+1, \'loss\': float(np.mean(losses)), \'validation\': validation,\n                  \'epoch_seconds\': time.perf_counter()-started}\n        history.append(record)\n        print(\'Epoch\', epoch+1, \'| loss\', round(record[\'loss\'], 4), \'| validation AP50\', round(validation[\'ap_50\'], 4), flush=True)\n        if validation[\'ap_50\'] > best_ap:\n            best_ap = validation[\'ap_50\']\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        write_json(project / \'results/training_history.json\', history)\n    model.load_state_dict(best_state)\n    # Evaluate the untouched test partition once after validation selects a model.\n    metrics = evaluate_detector(model, test_set, device)\n    metrics.update({\'partition\': \'test\', \'selection\': \'best validation AP50\',\n                    \'epochs\': epochs, \'training_images\': len(train_set),\n                    \'validation_best_ap50\': best_ap,\n                    \'dataset\': DATA_DOI, \'seed\': seed,\n                    \'architecture\': architecture,\n                    \'canadian_accuracy_measured\': False,\n                    \'torch\': str(torch.__version__), \'torchvision\': str(torchvision.__version__)})\n    model_path = project / \'results/models/vessel_detector.pt\'\n    model_path.parent.mkdir(parents=True, exist_ok=True)\n    # Half precision storage keeps the complete checkpoint below 50 MB.\n    compact = {k: v.half() if v.is_floating_point() else v for k, v in best_state.items()}\n    architecture_name = \'fasterrcnn_resnet18_fpn_small\' if architecture==\'resnet18\' else \'fasterrcnn_mobilenet_v3_large_fpn\'\n    torch.save({\'state_dict\': compact, \'architecture\': architecture_name,\n                \'input_size\': 256, \'class_names\': [\'background\', \'ship\'], \'metrics\': metrics}, model_path)\n    assert model_path.stat().st_size < 50_000_000\n    write_json(project / \'results/test_metrics.json\', metrics)\n    print(\'Checkpoint:\', model_path, \'| bytes:\', model_path.stat().st_size, flush=True)\n    print(json.dumps(metrics, indent=2), flush=True)\n    return model, device, metrics\n\n\ndef load_detector(path, device=None):\n    device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    saved = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if saved[\'architecture\'] not in [\'fasterrcnn_resnet18_fpn_small\',\'fasterrcnn_mobilenet_v3_large_fpn\']:\n        raise ValueError(\'Unsupported checkpoint architecture.\')\n    model = (build_detector(pretrained=False) if saved[\'architecture\']==\'fasterrcnn_resnet18_fpn_small\'\n             else build_mobilenet_detector(pretrained=False))\n    model.load_state_dict(saved[\'state_dict\'], strict=True)\n    return model.to(device).eval(), device, saved[\'metrics\']\n\n\ndef predict_rgb(model, device, rgb, threshold=0.3):\n    """Return x1,y1,x2,y2 boxes and scores in the supplied image pixel grid."""\n    array = np.asarray(rgb)\n    if array.ndim != 3 or array.shape[2] != 3 or array.dtype != np.uint8:\n        raise ValueError(\'Expected an HxWx3 uint8 RGB image.\')\n    tensor = torch.from_numpy(array.copy()).permute(2, 0, 1).float().to(device)/255\n    model.eval()\n    with torch.inference_mode():\n        prediction = model([tensor])[0]\n    keep = prediction[\'scores\'] >= threshold\n    return prediction[\'boxes\'][keep].cpu().numpy(), prediction[\'scores\'][keep].cpu().numpy()\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'ship_map.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Bounded-area Earth Engine -> Colab PyTorch -> geographic result workflow."""\nimport asyncio\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport html\nimport time\nfrom collections import OrderedDict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport ee\nimport geemap\nimport ipywidgets as widgets\nimport numpy as np\nimport rasterio\nimport requests\nimport torch\nimport torchvision\nfrom IPython.display import display, clear_output\nfrom PIL import Image, ImageDraw\nfrom pyproj import Transformer\nfrom ipyleaflet import DrawControl, GeoJSON, CircleMarker, LayerGroup\n\nfrom ship_backend import predict_rgb, write_json, checksum\n\nCOLLECTION = \'COPERNICUS/S2_HARMONIZED\'  # L1C, matching the training source.\nHALIFAX_BOUNDS = [-63.58, 44.615, -63.545, 44.64]\nPORT_SOURCE = \'https://github.com/smdg-org/Terminal-Code-List/blob/master/SMDG%20Terminal%20Code%20List.csv\'\nPORTS = [(\'Fairview Cove Container Terminal\', 44.667778, -63.627222),\n         (\'South End Container Terminal\', 44.628333, -63.5625)]\n\n\ndef validate_bounds(bounds, max_area_km2=16):\n    west, south, east, north = map(float, bounds)\n    if not all(math.isfinite(v) for v in [west, south, east, north]):\n        raise ValueError(\'Coordinates must be finite numbers.\')\n    if not (-141 <= west < east <= -52 and 41 <= south < north <= 84):\n        raise ValueError(\'Select a small area within the Canadian map extent.\')\n    width = (east-west)*111.32*math.cos(math.radians((south+north)/2))\n    height = (north-south)*111.32\n    if width*height > max_area_km2 or max(width, height) > 6:\n        raise ValueError(\'Area too large. Select at most 16 square kilometres and 6 km per side.\')\n    if min(width, height) < 0.1:\n        raise ValueError(\'Area too small. Select at least 100 m per side.\')\n    return [west, south, east, north]\n\n\nclass _RejectedObservation(ValueError):\n    """A scene failed a quality check and another candidate may work."""\n\n\ndef fetch_observation(bounds, start, end, scene_cloud_max=30):\n    """Retrieve one native 10 m observation; never use a multi-date composite."""\n    bounds = validate_bounds(bounds)\n    if not math.isfinite(scene_cloud_max) or not 0 <= scene_cloud_max <= 100:\n        raise ValueError(\'Scene cloud maximum must be between 0 and 100.\')\n    if datetime.fromisoformat(start) >= datetime.fromisoformat(end):\n        raise ValueError(\'The end date must be later than the start date (end exclusive).\')\n    started = time.perf_counter()\n    region = ee.Geometry.Rectangle(bounds, \'EPSG:4326\', False)\n    collection = (ee.ImageCollection(COLLECTION).filterBounds(region).filterDate(start, end)\n                  .filter(ee.Filter.lte(\'CLOUDY_PIXEL_PERCENTAGE\', scene_cloud_max))\n                  .sort(\'CLOUDY_PIXEL_PERCENTAGE\'))\n    candidates = collection.limit(12).getInfo()[\'features\']\n    search_seconds = time.perf_counter() - started\n    if not candidates:\n        raise ValueError(\'No matching images. Adjust dates or choose another area.\')\n    skipped = []\n    for candidate in candidates:\n        try:\n            rgb, valid, transform, crs, metadata = _fetch_candidate(\n                candidate, region, bounds, scene_cloud_max)\n        except _RejectedObservation as error:\n            skipped.append({\'scene\': candidate[\'id\'], \'reason\': str(error)})\n            continue\n        metadata[\'skipped_scenes\'] = skipped\n        metadata[\'scene_search_seconds\'] = search_seconds\n        metadata[\'imagery_total_seconds\'] = time.perf_counter() - started\n        return rgb, valid, transform, crs, metadata\n    reasons = \'; \'.join(dict.fromkeys(item[\'reason\'] for item in skipped))\n    raise ValueError(f\'No usable scene among {len(skipped)} candidates. {reasons} Adjust dates or area.\')\n\n\ndef _cloud_probability_image(image_id):\n    return (ee.ImageCollection(\'COPERNICUS/S2_CLOUD_PROBABILITY\')\n            .filter(ee.Filter.eq(\'system:index\', image_id.split(\'/\')[-1])))\n\n\ndef _fetch_candidate(selected, region, bounds, scene_cloud_max):\n    bands = [\'B4\', \'B3\', \'B2\']\n    raw = ee.Image(selected[\'id\'])\n    mask = raw.select(bands).mask().reduce(ee.Reducer.min())\n    coverage = mask.unmask(0, sameFootprint=False).reduceRegion(\n        ee.Reducer.mean(), region, 60, maxPixels=100000).getInfo()\n    if float(next(iter(coverage.values()), None) or 0) < 0.98:\n        raise _RejectedObservation(\'Area coverage below 98%.\')\n\n    cloud_collection = _cloud_probability_image(selected[\'id\'])\n    if cloud_collection.size().getInfo() == 0:\n        raise _RejectedObservation(\'No matching pixel-cloud assessment.\')\n    cloud = ee.Image(cloud_collection.first()).select(\'probability\').unmask(100)\n    projection = raw.select(\'B2\').projection().getInfo()\n    # Download RGB and cloud probability on exactly the same native grid.\n    download = raw.select(bands).addBands(cloud).toFloat()\n    download_started = time.perf_counter()\n    url = download.getDownloadURL({\'region\': region, \'scale\': 10,\n                                           \'crs\': projection[\'crs\'], \'format\': \'GEO_TIFF\'})\n    response = requests.get(url, timeout=120)\n    response.raise_for_status()\n    download_seconds = time.perf_counter() - download_started\n    with rasterio.MemoryFile(response.content) as memory:\n        with memory.open() as raster:\n            pixels = raster.read([1,2,3]).astype(np.float32)\n            available = np.all(raster.read_masks() > 0, axis=0) & np.any(pixels > 0, axis=0)\n            cloud_probability = raster.read(4)\n            valid = available & (cloud_probability < 65)\n            transform, crs = raster.transform, raster.crs.to_string()\n    clear_fraction = float(valid.sum()/max(available.sum(),1))\n    if clear_fraction < 0.8:\n        raise _RejectedObservation(\'Less than 80% of available pixels pass the pixel-cloud screen.\')\n    if pixels.shape[1]*pixels.shape[2] > 300000:\n        raise ValueError(\'Downloaded raster exceeds the interactive pixel limit.\')\n    # Approximate the training data\'s 1st/99th percentile stretch for usable scenes.\n    stats = raw.select(bands).reduceRegion(\n        ee.Reducer.percentile([1, 99]), raw.geometry(), 120,\n        bestEffort=True, maxPixels=2000000).getInfo()\n    lows = np.array([stats.get(b+\'_p1\', np.nan) for b in bands], dtype=np.float32)\n    highs = np.array([stats.get(b+\'_p99\', np.nan) for b in bands], dtype=np.float32)\n    if not np.isfinite(lows).all() or not np.isfinite(highs).all() or np.any(highs <= lows):\n        raise _RejectedObservation(\'Cannot determine a valid contrast stretch.\')\n    rgb = np.clip((pixels-lows[:, None, None])/(highs-lows)[:, None, None], 0, 1)\n    rgb = np.rint(rgb*255).astype(np.uint8).transpose(1, 2, 0)\n    rgb[~valid] = 0\n    metadata = {\'image_id\': selected[\'id\'], \'collection\': COLLECTION,\n                \'acquired_utc\': datetime.fromtimestamp(selected[\'properties\'][\'system:time_start\']/1000,\n                                                       timezone.utc).isoformat(),\n                \'scene_cloud_percent\': selected[\'properties\'][\'CLOUDY_PIXEL_PERCENTAGE\'],\n                \'scene_cloud_max\': scene_cloud_max,\n                \'bounds_wgs84\': bounds, \'pixel_size_m\': 10,\n                \'raster_crs\': crs, \'raster_transform\': list(transform)[:6],\n                \'raster_width\': rgb.shape[1], \'raster_height\': rgb.shape[0],\n                \'available_pixel_fraction\': float(available.mean()),\n                \'clear_pixel_fraction\': clear_fraction,\n                \'selected_scene_download_seconds\': download_seconds,\n                \'cloud_probability_collection\': \'COPERNICUS/S2_CLOUD_PROBABILITY\',\n                \'cloud_probability_max_exclusive\': 65,\n                \'preprocessing\': \'Scene-level 1/99 percentiles estimated at 120 m; RGB uint8\',\n                \'stretch_low\': lows.tolist(), \'stretch_high\': highs.tolist(),\n                \'quality_warning\': \'Pixel-cloud screening is imperfect; bright ships can be masked and land can produce false detections.\'}\n    return rgb, valid, transform, crs, metadata\n\n\ndef window_starts(size, chip=128, stride=96):\n    if size <= chip:\n        return [0]\n    return sorted(set(list(range(0, size-chip+1, stride)) + [size-chip]))\n\n\ndef detect_raster(model, device, rgb, valid, threshold=0.3):\n    """Overlapping native-resolution chips; global NMS removes repeat boxes."""\n    boxes, scores = [], []\n    height, width = rgb.shape[:2]\n    for y in window_starts(height):\n        for x in window_starts(width):\n            crop = rgb[y:y+128, x:x+128]\n            chip = np.zeros((128, 128, 3), np.uint8)\n            chip[:crop.shape[0], :crop.shape[1]] = crop\n            found, confidence = predict_rgb(model, device, chip, threshold)\n            for box, score in zip(found, confidence):\n                box = box + np.array([x, y, x, y])\n                box[[0, 2]] = box[[0, 2]].clip(0, width)\n                box[[1, 3]] = box[[1, 3]].clip(0, height)\n                cx, cy = int((box[0]+box[2])/2), int((box[1]+box[3])/2)\n                if box[2] > box[0] and box[3] > box[1] and valid[min(cy,height-1), min(cx,width-1)]:\n                    boxes.append(box.tolist())\n                    scores.append(float(score))\n    if not boxes:\n        return np.empty((0, 4)), np.empty(0)\n    keep = torchvision.ops.nms(torch.tensor(boxes), torch.tensor(scores), 0.3).numpy()\n    return np.array(boxes)[keep], np.array(scores)[keep]\n\n\ndef geographic_results(boxes, scores, transform, crs, metadata):\n    converter = Transformer.from_crs(crs, \'EPSG:4326\', always_xy=True)\n    features = []\n    for index, (box, score) in enumerate(zip(boxes, scores), 1):\n        x1, y1, x2, y2 = map(float, box)\n        corners = [transform*(x1,y1), transform*(x2,y1), transform*(x2,y2), transform*(x1,y2)]\n        ring = [list(converter.transform(x, y)) for x, y in corners]\n        ring.append(ring[0])\n        features.append({\'type\': \'Feature\', \'geometry\': {\'type\': \'Polygon\', \'coordinates\': [ring]},\n                         \'properties\': {\'detection_id\': index, \'class\': \'ship\', \'confidence\': float(score),\n                                        \'image_id\': metadata[\'image_id\'], \'acquired_utc\': metadata[\'acquired_utc\'],\n                                        \'pixel_box_xyxy\': [x1,y1,x2,y2]}})\n    return {\'type\': \'FeatureCollection\', \'features\': features}\n\n\ndef save_detection(project, rgb, boxes, scores, geojson, metadata):\n    project = Path(project)\n    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:12]\n    stem = \'ships_\' + metadata[\'acquired_utc\'][:10] + \'_\' + key\n    folder = project / \'results\'\n    folder.mkdir(parents=True, exist_ok=True)\n    write_json(folder / f\'{stem}.geojson\', geojson)\n    write_json(folder / f\'{stem}.json\', metadata)\n    with (folder / f\'{stem}.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n        writer = csv.writer(handle)\n        writer.writerow([\'detection_id\',\'class\',\'confidence\',\'longitude\',\'latitude\',\'image_id\',\'acquired_utc\'])\n        for feature in geojson[\'features\']:\n            ring = feature[\'geometry\'][\'coordinates\'][0][:-1]\n            lon, lat = np.array(ring).mean(axis=0)\n            prop = feature[\'properties\']\n            writer.writerow([prop[\'detection_id\'],\'ship\',prop[\'confidence\'],lon,lat,prop[\'image_id\'],prop[\'acquired_utc\']])\n    preview = Image.fromarray(rgb)\n    draw = ImageDraw.Draw(preview)\n    for box, score in zip(boxes, scores):\n        draw.rectangle(tuple(map(float, box)), outline=\'#ff3636\', width=2)\n        draw.text((float(box[0]), max(0., float(box[1])-11)), f\'{score:.2f}\', fill=\'#ff3636\')\n    preview.save(folder / f\'{stem}.png\')\n    print(\'Saved CSV, GeoJSON, PNG and metadata:\', stem)\n    return stem\n\n\ndef build_detection_app(project, model, device, metrics):\n    """Return the Colab application; model and Earth Engine are caller-owned."""\n    project = Path(project)\n    reference_features = []\n    with (project/\'data/halifax_port_reference.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as handle:\n        writer = csv.writer(handle)\n        writer.writerow([\'name\',\'latitude\',\'longitude\',\'source_url\',\'source_row_date\',\'record_type\'])\n        for name, lat, lon in PORTS:\n            writer.writerow([name,lat,lon,PORT_SOURCE,\'2020-01-24\',\'published reference point\'])\n            reference_features.append({\'type\':\'Feature\',\'geometry\':{\'type\':\'Point\',\'coordinates\':[lon,lat]},\n                                       \'properties\':{\'name\':name,\'source_url\':PORT_SOURCE,\n                                                     \'source_row_date\':\'2020-01-24\',\'record_type\':\'reference\'}})\n    write_json(project/\'data/halifax_port_reference.geojson\',{\'type\':\'FeatureCollection\',\'features\':reference_features})\n    Map = geemap.Map(center=[58, -96], zoom=3, ee_initialize=False,\n                     draw_ctrl=False, height=\'550px\')\n    Map.clear_layers()\n    Map.add_basemap(\'Esri.WorldGrayCanvas\')\n    status, output = widgets.Output(), widgets.Output()\n    fields = [widgets.FloatText(value=value, description=name, layout=widgets.Layout(width=\'240px\'))\n              for name, value in zip([\'West:\', \'South:\', \'East:\', \'North:\'], HALIFAX_BOUNDS)]\n    start = widgets.Text(value=\'2025-06-01\', description=\'From:\')\n    end = widgets.Text(value=\'2025-09-01\', description=\'To (excl.):\')\n    threshold = widgets.FloatSlider(value=0.55, min=0.05, max=0.95, step=0.05, description=\'Confidence:\')\n    selection_path = project/\'results/threshold_selection.json\'\n    if selection_path.exists():\n        threshold.value = float(json.loads(selection_path.read_text())[\'validation\'][\'threshold\'])\n    cloud_max = widgets.IntSlider(value=30, min=0, max=100, step=5, description=\'Cloud max %:\')\n    detect = widgets.Button(description=\'Detect ships\', button_style=\'primary\')\n    preview = widgets.Button(description=\'Load imagery\')\n    canada = widgets.Button(description=\'Canada overview\')\n    halifax = widgets.Button(description=\'Halifax example\')\n    pick = widgets.ToggleButton(value=False, description=\'Pick area on map\')\n    auto = widgets.Checkbox(value=False, description=\'Detect after moving map (zoom 15+)\')\n    ports = widgets.Checkbox(value=True, description=\'Show port/marina references\')\n    count = widgets.HTML(\'<b>Ship detections:</b> not run\')\n    details = widgets.HTML(\'Click a predicted box to inspect its confidence.\')\n    state = {\'last\': None, \'layers\': [], \'cache\': OrderedDict(), \'busy\': False, \'task\': None}\n\n    presets = {\n        \'Halifax (Atlantic)\': HALIFAX_BOUNDS,\n        \'Vancouver (Pacific)\': [-123.115,49.285,-123.08,49.31],\n        \'Montreal (St Lawrence)\': [-73.55,45.53,-73.515,45.555],\n        \'Hamilton (Great Lakes)\': [-79.875,43.270,-79.862,43.280],\n        \'Churchill (Hudson Bay)\': [-94.23,58.765,-94.195,58.79],\n        \'Iqaluit (Arctic)\': [-68.54,63.725,-68.505,63.75],\n        \'Prince Rupert (Pacific North)\': [-130.345,54.29,-130.31,54.315],\n        \'St Johns (Newfoundland)\': [-52.72,47.555,-52.685,47.58],\n        \'Simpson Point (Yukon)\': [-138.955,69.554,-138.877,69.584]}\n    preset = widgets.Dropdown(options=[\'Choose area\']+list(presets), description=\'Region:\')\n    from national_data import read_map_references\n    references = read_map_references(project)\n    province = widgets.Dropdown(options=[\'All\']+sorted({r[\'province\'] for r in references}), description=\'Province:\')\n    subtype = widgets.Dropdown(options=[\'All\']+sorted({r[\'subtype\'] for r in references}), description=\'Facility:\')\n    search = widgets.Text(description=\'Port name:\', placeholder=\'Search national references\')\n    port_select = widgets.Dropdown(options=[(\'Select a reference\', \'\')], description=\'Reference:\')\n    reference_count = widgets.HTML()\n    download = widgets.Dropdown(options=[\'GeoJSON\',\'CSV\',\'PNG\',\'Metadata\'], description=\'Export:\')\n    download_button = widgets.Button(description=\'Download current result\', disabled=True)\n\n    terminal_markers = []\n    for name, lat, lon in PORTS:\n        marker = CircleMarker(location=(lat,lon), radius=7, color=\'white\', fill_color=\'#bd3154\', fill_opacity=1)\n        marker.popup = widgets.HTML(\'<div style="color:#17212b;background:white"><b>\'+name+\'</b><br>\'\n                                   \'Published reference point, not a detection.<br>Source row: 2020-01-24<br>\'\n                                   \'<a href="\'+PORT_SOURCE+\'" target="_blank">SMDG reference list</a></div>\')\n        terminal_markers.append(marker)\n    port_layer = LayerGroup(layers=tuple(terminal_markers), name=\'Terminal references\')\n    Map.add_layer(port_layer)\n    ports.observe(lambda change: setattr(port_layer, \'layers\', tuple(terminal_markers) if change[\'new\'] else ()), names=\'value\')\n\n    national_layer = GeoJSON(data={\'type\':\'FeatureCollection\',\'features\':[]}, name=\'National port/marina references\',\n                            point_style={\'radius\':4,\'color\':\'#265f9e\',\'fillOpacity\':0.7})\n    Map.add_layer(national_layer)\n\n    def update_references(*_):\n        from national_data import filter_ports\n        rows = filter_ports(references, province.value, subtype.value, search.value)\n        reference_count.value = (f\'<b>References:</b> {len(rows):,} of {len(references):,}<br>\'\n                                \'Published facilities; coverage varies. Blue points are not detections.\') if references else (\n                                \'National inventory is not loaded. Run acquire_ports first and rebuild the dashboard.\')\n        port_select.options = [(\'Select a reference\',\'\')]+[(r[\'name\']+\' | \'+r[\'province\'],r[\'reference_id\']) for r in rows]\n        national_layer.data = {\'type\':\'FeatureCollection\',\'features\':[\n            {\'type\':\'Feature\',\'geometry\':{\'type\':\'Point\',\'coordinates\':[float(r[\'longitude\']),float(r[\'latitude\'])]},\n             \'properties\':{\'name\':r[\'name\'],\'province\':r[\'province\'],\'subtype\':r[\'subtype\'],\n                           \'provider\':r.get(\'provider\',\'\'),\'source_url\':r.get(\'source_url\',\'\'),\n                           \'record_type\':r.get(\'record_type\',\'published reference\'),\n                           \'validation_status\':r.get(\'validation_status\',\'\')}} for r in rows]} if ports.value else {\n                               \'type\':\'FeatureCollection\',\'features\':[]}\n    for widget in [province, subtype, search, ports]:\n        widget.observe(update_references,names=\'value\')\n    update_references()\n\n    def choose_preset(change):\n        if change[\'new\'] in presets:\n            bounds = presets[change[\'new\']]\n            for field,value in zip(fields,bounds):\n                field.value=value\n            Map.center=((bounds[1]+bounds[3])/2,(bounds[0]+bounds[2])/2)\n            Map.zoom=14\n    preset.observe(choose_preset,names=\'value\')\n\n    def choose_port(change):\n        row=next((r for r in references if r[\'reference_id\']==change[\'new\']),None)\n        if row:\n            lat,lon=float(row[\'latitude\']),float(row[\'longitude\'])\n            dx=.009/math.cos(math.radians(lat))\n            for field,value in zip(fields,[lon-dx,lat-.009,lon+dx,lat+.009]):\n                field.value=value\n            Map.center,Map.zoom=(lat,lon),14\n    port_select.observe(choose_port,names=\'value\')\n\n    def clear_results():\n        for layer in state[\'layers\']:\n            if layer in Map.layers:\n                Map.remove_layer(layer)\n        state[\'layers\'] = []\n        state[\'last\'] = None\n        state.pop(\'output_stem\', None)\n        download_button.disabled = True\n        count.value = \'<b>Ship detections:</b> not run for current settings\'\n        details.value = \'Click a predicted box to inspect its confidence.\'\n        with output:\n            clear_output(wait=False)\n\n    def changed(_):\n        if not state[\'busy\']:\n            clear_results()\n\n    for field in fields + [start, end, threshold, cloud_max]:\n        field.observe(changed, names=\'value\')\n\n    def run(inference=True):\n        if state[\'busy\']:\n            return\n        state[\'busy\'] = True\n        detect.disabled = preview.disabled = True\n        clear_results()\n        with status:\n            clear_output(wait=True)\n            print(\'Loading the selected satellite observation...\')\n        try:\n            if inference and model is None:\n                raise ValueError(\'Restore the preserved checkpoint before Detect ships. References and saved results remain available.\')\n            bounds = validate_bounds([field.value for field in fields])\n            key = (tuple(bounds), start.value, end.value, cloud_max.value)\n            cache_hit = key in state[\'cache\']\n            if not cache_hit:\n                state[\'cache\'][key] = fetch_observation(bounds, start.value, end.value, cloud_max.value)\n                if len(state[\'cache\']) > 4:\n                    state[\'cache\'].popitem(last=False)\n            rgb, valid, transform, crs, source_metadata = state[\'cache\'][key]\n            metadata = dict(source_metadata)\n            metadata[\'imagery_cache_hit\'] = cache_hit\n            map_started = time.perf_counter()\n            west, south, east, north = bounds\n            selection = GeoJSON(data={\'type\':\'Feature\',\'properties\':{},\'geometry\':{\'type\':\'Polygon\',\n                                \'coordinates\':[[[west,south],[east,south],[east,north],[west,north],[west,south]]]}},\n                                name=\'Selected area\', style={\'color\':\'#ff9900\',\'weight\':2,\'fillOpacity\':0})\n            Map.add_layer(selection)\n            state[\'layers\'].append(selection)\n            # Draw the same RGB pixels used by inference, not a different composite.\n            # GeoTIFF pixel grid is projected; use geemap\'s EE visualization for\n            # correctly placed context, and GeoJSON for exact detection geometry.\n            raw = ee.Image(metadata[\'image_id\']).select([\'B4\',\'B3\',\'B2\'])\n            low = ee.Image.constant(metadata[\'stretch_low\']).rename([\'B4\',\'B3\',\'B2\'])\n            high = ee.Image.constant(metadata[\'stretch_high\']).rename([\'B4\',\'B3\',\'B2\'])\n            cloud = ee.Image(_cloud_probability_image(metadata[\'image_id\']).first()).select(\'probability\')\n            rgb_valid = raw.mask().reduce(ee.Reducer.min()).And(raw.gt(0).reduce(ee.Reducer.max()))\n            visible = (raw.subtract(low).divide(high.subtract(low)).clamp(0,1)\n                       .updateMask(rgb_valid.And(cloud.unmask(100).lt(65)))\n                       .clip(ee.Geometry.Rectangle(bounds)))\n            Map.addLayer(visible, {\'min\':0,\'max\':1}, \'Selected Sentinel-2 observation\')\n            state[\'layers\'].append(Map.layers[-1])\n            metadata[\'map_layer_setup_seconds\'] = time.perf_counter() - map_started\n            if inference:\n                started = time.perf_counter()\n                boxes, scores = detect_raster(model, device, rgb, valid, threshold.value)\n                result = geographic_results(boxes, scores, transform, crs, metadata)\n                metadata.update({\'threshold\': threshold.value, \'detection_count\': len(boxes),\n                                 \'inference_seconds\': time.perf_counter()-started,\n                                 \'model\': \'Fine-tuned Faster R-CNN\',\n                                 \'test_ap50\': metrics.get(\'ap_50\'), \'canadian_accuracy_measured\': False,\n                                 \'run_utc\': datetime.now(timezone.utc).isoformat(),\n                                 \'model_checkpoint_sha256\': checksum(project/\'results/models/vessel_detector.pt\', \'sha256\')})\n                layer = GeoJSON(data=result, name=\'Predicted ships\', style={\'color\':\'#ff3636\',\'weight\':2,\'fillOpacity\':0.1})\n                def clicked(**event):\n                    feature = event.get(\'feature\', {})\n                    prop = feature.get(\'properties\', event.get(\'properties\', {}))\n                    if \'confidence\' in prop:\n                        details.value = (f"Prediction #{prop[\'detection_id\']} | ship | confidence {prop[\'confidence\']:.3f}"\n                                         f" | acquired {metadata[\'acquired_utc\']}")\n                layer.on_click(clicked)\n                Map.add_layer(layer)\n                state[\'layers\'].append(layer)\n                state[\'last\'] = (result, metadata)\n                count.value = f\'<b>Predicted ships:</b> {len(boxes)} | confidence ≥ {threshold.value:.2f}\'\n                with output:\n                    clear_output(wait=True)\n                    saved_stem = save_detection(project, rgb, boxes, scores, result, metadata)\n                    state[\'output_stem\'] = saved_stem\n                    download_button.disabled = False\n                    with Image.open(project/\'results\'/f\'{saved_stem}.png\') as saved_preview:\n                        saved_preview.thumbnail((600,600))\n                        display(saved_preview.copy())\n            with status:\n                clear_output(wait=True)\n                print(\'Acquired:\', metadata[\'acquired_utc\'])\n                print(\'Scene:\', metadata[\'image_id\'])\n                print(f"Clear coverage: {metadata[\'clear_pixel_fraction\']:.0%} of available pixels.")\n                skipped_count = len(metadata.get(\'skipped_scenes\', []))\n                if skipped_count:\n                    print(f\'Skipped {skipped_count} other scene(s) that failed quality checks.\')\n                print(\'10 m imagery. Results are experimental; Canadian accuracy is not yet validated.\')\n                if not inference:\n                    print(\'Imagery loaded. Click Detect ships to run the model.\')\n        except Exception as error:\n            with status:\n                clear_output(wait=True)\n                # Do not print signed download URLs or credential-bearing errors.\n                print(str(error) if isinstance(error, ValueError) else \'Request failed: \'+type(error).__name__+\'. Retry or choose another area.\')\n        finally:\n            state[\'busy\'] = False\n            detect.disabled = preview.disabled = False\n\n    def go_halifax(_):\n        for field, value in zip(fields, HALIFAX_BOUNDS):\n            field.value = value\n        Map.center, Map.zoom = (44.628,-63.5625), 14\n\n    draw = DrawControl(rectangle={\'shapeOptions\': {\'color\':\'#ff9900\'}},\n                       polygon={}, polyline={}, circle={}, circlemarker={}, marker={})\n\n    def drawn(_, action, geo_json):\n        clear_results()\n        if action == \'deleted\':\n            return\n        coords = np.asarray(geo_json[\'geometry\'][\'coordinates\'][0])\n        bounds = [coords[:,0].min(), coords[:,1].min(), coords[:,0].max(), coords[:,1].max()]\n        for field, value in zip(fields, bounds):\n            field.value = float(value)\n        with status:\n            clear_output(wait=True)\n            print(\'Selection updated. Click Detect ships.\')\n\n    draw.on_draw(drawn)\n    Map.add_control(draw)\n\n    def map_clicked(**event):\n        if event.get(\'type\') != \'click\' or Map.zoom < 12 or not pick.value:\n            return\n        lat, lon = event[\'coordinates\']\n        dx = 0.009/math.cos(math.radians(lat))\n        for field, value in zip(fields, [lon-dx,lat-0.009,lon+dx,lat+0.009]):\n            field.value = value\n        pick.value = False\n\n    Map.on_interaction(map_clicked)\n\n    def moved(_):\n        if state[\'task\'] and not state[\'task\'].done():\n            state[\'task\'].cancel()\n        if not auto.value or Map.zoom < 15 or state[\'busy\']:\n            return\n\n        async def after_pause():\n            await asyncio.sleep(1.)\n            if not auto.value or Map.zoom < 15:\n                return\n            (south, west), (north, east) = Map.bounds\n            for field, value in zip(fields, [west,south,east,north]):\n                field.value = value\n            run(True)\n        state[\'task\'] = asyncio.get_running_loop().create_task(after_pause())\n\n    Map.observe(moved, names=\'bounds\')\n    detect.on_click(lambda _: run(True))\n    preview.on_click(lambda _: run(False))\n    canada.on_click(lambda _: (setattr(Map,\'center\',(58,-96)), setattr(Map,\'zoom\',3)))\n    halifax.on_click(go_halifax)\n    def export_current(_):\n        from google.colab import files\n        suffix={\'GeoJSON\':\'.geojson\',\'CSV\':\'.csv\',\'PNG\':\'.png\',\'Metadata\':\'.json\'}[download.value]\n        if state.get(\'output_stem\'):\n            files.download(str(project/\'results\'/(state[\'output_stem\']+suffix)))\n    download_button.on_click(export_current)\n\n    saved_paths = sorted((project/\'results\').glob(\'ships_*.geojson\'))\n    saved = widgets.Dropdown(options=[(\'Select saved prediction\',\'\')]+[(p.stem,p.stem) for p in saved_paths],description=\'Saved run:\')\n    def show_saved(change):\n        if not change[\'new\']:\n            return\n        clear_results()\n        stem=change[\'new\']\n        data=json.loads((project/\'results\'/f\'{stem}.geojson\').read_text())\n        metadata=json.loads((project/\'results\'/f\'{stem}.json\').read_text())\n        layer=GeoJSON(data=data,name=\'Saved experimental predictions\',style={\'color\':\'#ff3636\',\'weight\':2,\'fillOpacity\':.1})\n        Map.add_layer(layer)\n        state[\'layers\'].append(layer)\n        # Keep last=None: archived output must not masquerade as current-settings inference.\n        state[\'output_stem\']=stem\n        download_button.disabled=False\n        count.value=f\'<b>Saved predictions:</b> {len(data["features"])} | \'+html.escape(metadata[\'acquired_utc\'])\n        details.value=\'Archived run; dates, area and confidence come from its metadata, not current controls.\'\n        bounds=metadata.get(\'bounds_wgs84\')\n        if bounds:\n            Map.center=((bounds[1]+bounds[3])/2,(bounds[0]+bounds[2])/2)\n            Map.zoom=14\n        with output:\n            clear_output(wait=True)\n            image_path=project/\'results\'/f\'{stem}.png\'\n            if image_path.exists():\n                with Image.open(image_path) as image:\n                    image.thumbnail((700,550))\n                    display(image.copy())\n    saved.observe(show_saved,names=\'value\')\n    sidebar=widgets.VBox([widgets.HTML(\'<h4>Explore Canada</h4>\'),preset,province,subtype,search,port_select,\n                          ports,reference_count,widgets.HTML(\'<h4>Ship detection</h4>\'),start,end,threshold,cloud_max,\n                          widgets.HTML(\'End date is exclusive. Native 10 m input; small boats may be unresolved.\'),\n                          *fields,widgets.HBox([preview,detect]),pick,auto,\n                          widgets.HTML(\'<h4>Results and exports</h4>\'),saved,download,download_button],\n                         layout=widgets.Layout(width=\'340px\',min_width=\'300px\',padding=\'8px\'))\n    main=widgets.VBox([widgets.HBox([canada,halifax]),count,details,status,Map,output],layout=widgets.Layout(flex=\'1\'))\n    panel = widgets.VBox([widgets.HTML(\'<h3>Ships and ports in Canada</h3>\'\n                         \'<p>Experimental ship detector | Port and marina references | Canadian accuracy pending</p>\'),\n                         widgets.HBox([sidebar,main],layout=widgets.Layout(width=\'100%\',align_items=\'flex-start\'))])\n    return panel, {\'map\':Map, \'run\':run, \'state\':state, \'fields\':fields,\n                   \'threshold\':threshold, \'go_halifax\':go_halifax, \'auto\':auto,\n                   \'draw_callback\':drawn, \'port_layer\':port_layer, \'ports\':ports,\n                   \'pick\':pick,\'details\':details,\'preset\':preset,\'province\':province,\'subtype\':subtype,\n                   \'search\':search,\'port_select\':port_select,\'national_layer\':national_layer,\n                   \'cloud_max\':cloud_max,\'saved\':saved,\'download_button\':download_button}\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'national_data.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""National reference acquisition in Google Colab; never ship ground truth."""\nimport hashlib\nimport io\nimport json\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nODI_PAGE = \'https://www150.statcan.gc.ca/n1/pub/34-26-0003/342600032023001-eng.htm\'\nODI_URL = \'https://www150.statcan.gc.ca/pub/34-26-0003/2023001/zip/ODI_v2_ports_marinas.zip\'\nLICENCE = \'Open Government Licence - Canada\'\n\n\ndef read_map_references(project):\n    """Combine the ODI snapshot with separately attributed map references."""\n    import csv\n    path = Path(project) / \'data\' / \'canada_ports_reference.csv\'\n    if not path.exists():\n        return []\n    with path.open(encoding=\'utf-8\', newline=\'\') as handle:\n        references = list(csv.DictReader(handle))\n    supplement = path.with_name(\'canada_facility_supplements.csv\')\n    if supplement.exists():\n        with supplement.open(encoding=\'utf-8\', newline=\'\') as handle:\n            references.extend(csv.DictReader(handle))\n    return references\n\n\ndef acquire_ports(project):\n    """Retain provider metadata and geometries; derive map points in EPSG:4326."""\n    import geopandas as gpd\n    import pandas as pd\n    import requests\n    project = Path(project)\n    raw = project/\'data/raw/odi_ports_v2\'\n    raw.mkdir(parents=True, exist_ok=True)\n    response = requests.get(ODI_URL, timeout=120)\n    response.raise_for_status()\n    payload = response.content\n    with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n        # Extract data/metadata only, with no archive-provided executable code.\n        for member in archive.infolist():\n            target = (raw/member.filename).resolve()\n            if not target.is_relative_to(raw.resolve()):\n                raise ValueError(\'Unsafe source archive path.\')\n            if not member.is_dir() and target.suffix.lower() in {\'.gpkg\', \'.csv\', \'.pdf\'}:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(archive.read(member))\n    packages = list(raw.rglob(\'*.gpkg\'))\n    if len(packages) != 1:\n        raise ValueError(\'Expected one ports/marinas GeoPackage; inspect the source layout.\')\n    frame = gpd.read_file(packages[0])\n    if frame.crs is None:\n        raise ValueError(\'Source reference system is missing.\')\n    original_columns = list(frame.columns)\n    def column(*names):\n        lookup = {c.lower(): c for c in frame.columns}\n        for name in names:\n            if name in lookup:\n                return lookup[name]\n        raise ValueError(\'Review ODI schema; missing one of \'+repr(names))\n    # Source field aliases are explicit. Schema changes fail visibly.\n    name = column(\'name\', \'name_en\', \'infrastructure_name\')\n    province = column(\'province_territory\', \'prov_terr\', \'province\', \'prname\', \'pruid\')\n    subtype = column(\'sub_type\', \'infrastructure_subtype\', \'infra_type\', \'infra_subtype\', \'type\', \'subtype\')\n    source_id = column(\'unique_id\', \'id\', \'uid\')\n    frame = frame[frame.geometry.notna() & ~frame.geometry.is_empty].copy()\n    # Representative points are display anchors, not surveyed port locations.\n    points = frame.to_crs(3347).geometry.representative_point().to_crs(4326)\n    normal = pd.DataFrame({\n        \'reference_id\': frame[source_id].astype(str), \'name\': frame[name].fillna(\'Unnamed reference\').astype(str),\n        \'province\': frame[province].fillna(\'Unknown\').astype(str),\n        \'subtype\': frame[subtype].fillna(\'Unknown\').astype(str),\n        \'longitude\': points.x, \'latitude\': points.y,\n        \'source_url\': ODI_PAGE, \'record_type\': \'published infrastructure reference\',\n        \'validation_status\': \'provider record; not ship/port detection ground truth\'})\n    for field in [\'provider\',\'source_id\',\'source_class\',\'ownership\',\'jurisdiction\',\'un_locode\']:\n        if field in frame.columns:\n            normal[field]=frame[field].fillna(\'Unknown\').astype(str)\n    if normal.reference_id.duplicated().any():\n        raise ValueError(\'Duplicate ODI identifiers require review.\')\n    normal.to_csv(project/\'data/canada_ports_reference.csv\', index=False)\n    features = [{\'type\':\'Feature\', \'geometry\':{\'type\':\'Point\',\'coordinates\':[r[\'longitude\'],r[\'latitude\']]},\n                 \'properties\':{k:v for k,v in r.items() if k not in {\'longitude\',\'latitude\'}}}\n                for r in normal.to_dict(\'records\')]\n    (project/\'data/canada_ports_reference.geojson\').write_text(\n        json.dumps({\'type\':\'FeatureCollection\',\'features\':features}, indent=2), encoding=\'utf-8\')\n    report = {\'source_url\': ODI_PAGE, \'download_url\': ODI_URL, \'licence\': LICENCE,\n              \'attribution\':\'Contains information licensed under the Open Government Licence - Canada.\',\n              \'retrieved_utc\':datetime.now(timezone.utc).isoformat(),\n              \'archive_sha256\':hashlib.sha256(payload).hexdigest(), \'source_version\':\'ODI v2\',\n              \'source_collection_period\':\'October 2023 to June 2024\', \'exact_record_acquisition_dates\':None,\n              \'records\':len(normal), \'province_counts\':normal.province.value_counts().to_dict(),\n              \'provinces_territories_without_records\':sorted(set([\'AB\',\'BC\',\'MB\',\'NB\',\'NL\',\'NS\',\'NT\',\'NU\',\'ON\',\'PE\',\'QC\',\'SK\',\'YT\'])-set(normal.province)),\n              \'subtype_counts\':normal.subtype.value_counts().to_dict(), \'source_columns\':original_columns,\n              \'metadata_files\':[str(p.relative_to(project)) for p in raw.rglob(\'*\') if p.suffix in {\'.csv\',\'.pdf\'}],\n              \'limitations\':[\'Selected infrastructure; geographic completeness varies by source and licence.\',\n                             \'Ports and marinas include different facility types; filter explicitly.\',\n                             \'Derived representative map points are not port boundaries or ship labels.\',\n                             \'No assumption of current operating status or exhaustive Canadian coverage.\']}\n    (project/\'data/canada_ports_source.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    return normal, report\n\n\ndef filter_ports(records, province=\'All\', subtype=\'All\', query=\'\'):\n    """Pure reference filter shared by Colab and a future host UI."""\n    query = query.strip().casefold()\n    return [r for r in records if (province == \'All\' or r[\'province\'] == province)\n            and (subtype == \'All\' or r[\'subtype\'] == subtype)\n            and (not query or query in str(r[\'name\']).casefold())]\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'dashboard_adapter.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Caller-owned Colab backend and widget factory for a future course host."""\nfrom dataclasses import dataclass\nfrom datetime import date\nimport math\n\n\n@dataclass(frozen=True)\nclass DetectionRequest:\n    bounds: tuple\n    start: str\n    end: str\n    confidence: float = 0.55\n    scene_cloud_max: float = 30\n\n    def validate(self):\n        from ship_map import validate_bounds\n        validate_bounds(self.bounds)\n        if date.fromisoformat(self.start) >= date.fromisoformat(self.end):\n            raise ValueError(\'End date must follow start date; end is exclusive.\')\n        if not math.isfinite(self.confidence) or not 0.05 <= self.confidence <= 0.95:\n            raise ValueError(\'Confidence must be between 0.05 and 0.95.\')\n        if not math.isfinite(self.scene_cloud_max) or not 0 <= self.scene_cloud_max <= 100:\n            raise ValueError(\'Scene cloud maximum must be between 0 and 100.\')\n        return self\n\n\ndef run_detection(project, model, device, request):\n    """Same Python call for standalone widgets and a host; no HTTP server/tunnel."""\n    from pathlib import Path\n    from datetime import datetime, timezone\n    import time\n    from ship_backend import checksum\n    from ship_map import fetch_observation, detect_raster, geographic_results, save_detection\n    request.validate()\n    if model is None:\n        raise ValueError(\'Restore the preserved checkpoint before running detection.\')\n    rgb, valid, transform, crs, metadata = fetch_observation(\n        request.bounds, request.start, request.end, request.scene_cloud_max)\n    started = time.perf_counter()\n    boxes, scores = detect_raster(model, device, rgb, valid, request.confidence)\n    metadata.update(threshold=request.confidence, detection_count=len(boxes),\n                    scene_cloud_max=request.scene_cloud_max, inference_seconds=time.perf_counter()-started,\n                    model_checkpoint_sha256=checksum(Path(project)/\'results/models/vessel_detector.pt\',\'sha256\'),\n                    run_utc=datetime.now(timezone.utc).isoformat(), canadian_accuracy_measured=False,\n                    validation_status=\'experimental predictions; not validated ground truth\')\n    geojson = geographic_results(boxes, scores, transform, crs, metadata)\n    stem = save_detection(project, rgb, boxes, scores, geojson, metadata)\n    return {\'geojson\':geojson, \'metadata\':metadata, \'output_stem\':stem}\n\n\ndef create_project_view(project, model=None, device=None, metrics=None):\n    """A course project button should call this factory to reload the entire map."""\n    from ship_map import build_detection_app\n    return build_detection_app(project, model, device, metrics or {})\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'canadian_validation.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Canadian review eligibility and locked geographic split plan. Run in Colab."""\nimport csv\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n# Assign regions BEFORE new label review or model experiments. Both Halifax\n# seasons stay together. Aerial imagery is a distinct domain, never mixed into S2.\nCASE_PLAN = [\n    (\'halifax_summer\',\'Atlantic\',\'Halifax\',\'train\',\'sentinel2_l1c_10m\'),\n    (\'halifax_autumn\',\'Atlantic\',\'Halifax\',\'train\',\'sentinel2_l1c_10m\'),\n    (\'vancouver\',\'Pacific\',\'Vancouver\',\'val\',\'sentinel2_l1c_10m\'),\n    (\'montreal\',\'St Lawrence\',\'Montreal\',\'test\',\'sentinel2_l1c_10m\'),\n    (\'hamilton\',\'Great Lakes\',\'Hamilton\',\'train\',\'ontario_aerial\'),\n    (\'toronto\',\'Great Lakes\',\'Toronto\',\'val\',\'ontario_aerial\'),\n    (\'kingston\',\'Great Lakes\',\'Kingston\',\'test\',\'ontario_aerial\'),\n    (\'churchill\',\'Hudson Bay\',\'Churchill\',\'test\',\'sentinel2_l1c_10m\'),\n    (\'iqaluit\',\'Arctic\',\'Iqaluit\',\'test\',\'sentinel2_l1c_10m\'),\n    (\'prince_rupert\',\'Pacific North\',\'Prince Rupert\',\'train\',\'sentinel2_l1c_10m\'),\n    (\'st_johns\',\'Newfoundland\',\'St Johns\',\'val\',\'sentinel2_l1c_10m\')]\n\n\ndef checked_annotation(label):\n    """A draft/empty template cannot become a negative example accidentally."""\n    if label.get(\'annotation_status\') != \'reviewer_checked\':\n        return False\n    if label.get(\'flags\',{}).get(\'review_required\',True):\n        return False\n    if not label.get(\'reviewer\',\'\').strip() or not label.get(\'reviewed_utc\'):\n        return False\n    width,height=label.get(\'imageWidth\',0),label.get(\'imageHeight\',0)\n    if not label.get(\'shapes\') and not label.get(\'negative_confirmed\',False):\n        return False\n    for shape in label.get(\'shapes\',[]):\n        if shape.get(\'label\')!=\'ship\' or shape.get(\'shape_type\')!=\'rectangle\':\n            return False\n        try:\n            (x1,y1),(x2,y2)=shape[\'points\']\n            if not (0<=x1<x2<=width and 0<=y1<y2<=height):\n                return False\n        except (KeyError,TypeError,ValueError):\n            return False\n    return width>0 and height>0\n\n\ndef build_review_manifest(project):\n    project=Path(project)\n    rows=[]\n    for case,region,group,split,domain in CASE_PLAN:\n        folder=\'aerial_review\' if domain==\'ontario_aerial\' else \'canadian_review\'\n        annotation=project/f\'data/{folder}/{case}.json\'\n        image=annotation.with_suffix(\'.png\')\n        label=json.loads(annotation.read_text()) if annotation.exists() else {}\n        eligible=image.exists() and checked_annotation(label)\n        rows.append(dict(case=case,region=region,geographic_group=group,split=split,domain=domain,\n                         image_path=str(image.relative_to(project)),annotation_path=str(annotation.relative_to(project)),\n                         review_status=label.get(\'annotation_status\',\'not_acquired\'),eligible=eligible,\n                         boxes=len(label.get(\'shapes\',[])) if eligible else \'\',\n                         image_sha256=hashlib.sha256(image.read_bytes()).hexdigest() if image.exists() else \'\',\n                         source_scene=label.get(\'source\',{}).get(\'image_id\',\'\')))\n    # Fail closed if identical imagery or scene is assigned to different splits.\n    for key in [\'geographic_group\',\'image_sha256\',\'source_scene\']:\n        assigned={}\n        for row in rows:\n            value=row[key]\n            if value and value in assigned and assigned[value]!=row[\'split\']:\n                raise ValueError(\'Canadian split leakage: \'+key)\n            if value:\n                assigned[value]=row[\'split\']\n    destination=project/\'data/canadian_review_manifest.csv\'\n    with destination.open(\'w\',newline=\'\',encoding=\'utf-8\') as handle:\n        writer=csv.DictWriter(handle,fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    report={\'generated_utc\':datetime.now(timezone.utc).isoformat(),\n            \'eligible_images\':sum(r[\'eligible\'] for r in rows),\n            \'planned_cases\':len(rows),\'canadian_accuracy_measured\':False,\n            \'status\':\'Review queue only; do not claim representative national accuracy.\',\n            \'test_policy\':\'Previously inspected predictions make existing cases exploratory. Acquire new scenes for final independent testing.\',\n            \'requirements\':[\'Freeze acquisition IDs and geographic groups before training.\',\n                            \'Add independent scenes, seasons and negatives in every region; a few harbours are insufficient.\',\n                            \'Train only on train; select epochs/thresholds only on val; unlock new test scenes once.\',\n                            \'Do not infer exact acquisition dates from aerial mosaic collection years.\',\n                            \'Use separate compatible training and evaluation for aerial and 10 m imagery.\']}\n    (project/\'results/canadian_review_readiness.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    return rows,report\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'review_annotations.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Colab image-and-box review; only an explicit reviewer can mark labels checked."""\nimport json\nfrom pathlib import Path\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\nfrom PIL import Image, ImageDraw\n\n\ndef annotation_review(project):\n    root = Path(project)\n    choices = sorted((root/\'data/canadian_review\').glob(\'*.png\')) + sorted((root/\'data/aerial_review\').glob(\'*.png\'))\n    pilot_image = root/\'data/canadian_pilot/saint_john_pilot.png\'\n    if pilot_image.with_suffix(\'.json\').is_file() and pilot_image.is_file():\n        choices.insert(0, pilot_image)\n    if not choices:\n        raise ValueError(\'Run the Canadian imagery cells first.\')\n    select = widgets.Dropdown(options=[(p.stem,str(p)) for p in choices],description=\'Image:\')\n    coords = [widgets.IntText(value=0,description=s,layout=widgets.Layout(width=\'170px\')) for s in [\'x1\',\'y1\',\'x2\',\'y2\']]\n    add=widgets.Button(description=\'Add ship box\')\n    undo=widgets.Button(description=\'Undo last box\')\n    save=widgets.Button(description=\'Save checked labels\',button_style=\'success\')\n    draft=widgets.Button(description=\'Save draft (unreviewed)\')\n    negative=widgets.Checkbox(description=\'Reviewed image contains no ships\',value=False,indent=False)\n    notes=widgets.Textarea(description=\'Review notes:\',placeholder=\'Ambiguities, cloud/ice, dry-stored boats or resolution limitations\')\n    reviewer=widgets.Text(description=\'Reviewer:\')\n    checked=widgets.Checkbox(description=\'All visible ships checked; ambiguous objects excluded\',value=False,indent=False)\n    blind=widgets.Checkbox(description=\'I drew these boxes before seeing model predictions\',value=False,indent=False)\n    out=widgets.Output()\n    state={\'shapes\':[]}\n\n    def show(*_):\n        with out:\n            clear_output(wait=True)\n            image=Image.open(select.value).convert(\'RGB\')\n            draw=ImageDraw.Draw(image)\n            for shape in state[\'shapes\']:\n                draw.rectangle(tuple(shape[\'points\'][0]+shape[\'points\'][1]),outline=\'red\',width=3)\n            print(\'Coordinates use the original image grid:\',image.size,\'| boxes:\',len(state[\'shapes\']))\n            image.thumbnail((1000,750))\n            display(image)\n\n    def change(_):\n        label=Path(select.value).with_suffix(\'.json\')\n        draft_path=Path(select.value).with_suffix(\'.draft.json\')\n        loaded=json.loads((draft_path if draft_path.exists() else label).read_text()) if label.exists() else {}\n        state[\'shapes\']=loaded.get(\'shapes\',[])\n        checked.value=False\n        negative.value=False\n        blind.value=False\n        blind.layout.display=\'\' if Path(select.value) == pilot_image else \'none\'\n        notes.value=loaded.get(\'review_notes\',\'\')\n        show()\n\n    def add_box(_):\n        x1,y1,x2,y2=[w.value for w in coords]\n        width,height=Image.open(select.value).size\n        if not (0<=x1<x2<=width and 0<=y1<y2<=height):\n            with out:\n                print(\'Box must have positive size and remain inside the original image grid.\')\n            return\n        state[\'shapes\'].append(dict(label=\'ship\',points=[[x1,y1],[x2,y2]],group_id=None,shape_type=\'rectangle\',flags={}))\n        checked.value=False\n        negative.value=False\n        show()\n\n    def remove(_):\n        if state[\'shapes\']:\n            state[\'shapes\'].pop()\n        checked.value=False\n        show()\n\n    def save_labels(_):\n        if not reviewer.value.strip() or not checked.value:\n            with out:\n                print(\'Enter the reviewer name and confirm the image has been checked before saving.\')\n            return\n        if not state[\'shapes\'] and not negative.value:\n            with out:\n                print(\'For an empty checked label, explicitly confirm that the reviewed image contains no ships.\')\n            return\n        from datetime import datetime,timezone\n        path=Path(select.value).with_suffix(\'.json\')\n        label=json.loads(path.read_text())\n        if Path(select.value) == pilot_image:\n            prior_result = (root/\'results/saint_john_pilot_metrics.json\').exists()\n            label[\'reviewed_before_predictions\'] = bool(blind.value and not prior_result)\n        history=path.parent/\'review_history\'\n        history.mkdir(exist_ok=True)\n        (history/(path.stem+\'_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%f\')+\'.json\')).write_text(path.read_text(),encoding=\'utf-8\')\n        label.update(shapes=state[\'shapes\'],flags={**label.get(\'flags\',{}),\'review_required\':False},annotation_status=\'reviewer_checked\',\n                     review_notes=notes.value,negative_confirmed=negative.value if not state[\'shapes\'] else False,\n                     reviewer=reviewer.value.strip(),reviewed_utc=datetime.now(timezone.utc).isoformat())\n        path.write_text(json.dumps(label,indent=2)+\'\\n\',encoding=\'utf-8\')\n        with out:\n            print(\'Saved:\',path.name,\'| checked ship boxes:\',len(state[\'shapes\']))\n\n    def save_draft(_):\n        path=Path(select.value).with_suffix(\'.draft.json\')\n        original=json.loads(Path(select.value).with_suffix(\'.json\').read_text())\n        original.update(shapes=state[\'shapes\'],annotation_status=\'draft_unreviewed\',\n                        flags={**original.get(\'flags\',{}),\'review_required\':True},review_notes=notes.value)\n        path.write_text(json.dumps(original,indent=2)+\'\\n\',encoding=\'utf-8\')\n        with out:\n            print(\'Saved draft separately; excluded from training and evaluation:\',path.name)\n\n    select.observe(change,names=\'value\')\n    add.on_click(add_box)\n    undo.on_click(remove)\n    save.on_click(save_labels)\n    draft.on_click(save_draft)\n    change(None)\n    return widgets.VBox([widgets.HTML(\'<b>Canadian annotation review</b><br>Draw tight axis-aligned boxes using original pixel coordinates. Do not label docks, wakes or uncertain objects as ships. Aerial imagery must be trained and evaluated separately from the 10 m model.\'),\n                         select,widgets.HBox(coords),widgets.HBox([add,undo,draft]),notes,reviewer,checked,negative,blind,save,out])\n', encoding='utf-8')


In [ ]:
module_path = PROJECT_DIR / 'code' / 'test_canada_dashboard.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Integration/regression checks to execute in Google Colab, without EE downloads."""\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\n\nclass CanadaDashboardChecks(unittest.TestCase):\n    def test_scene_selection_tries_next_candidate(self):\n        from ship_map import fetch_observation, HALIFAX_BOUNDS, _RejectedObservation\n\n        candidates = [{\'id\': \'missing-cloud\'}, {\'id\': \'too-cloudy\'}, {\'id\': \'clear\'}]\n        with patch(\'ship_map.ee\') as ee, patch(\n                \'ship_map._fetch_candidate\') as fetch:\n            ee.ImageCollection.return_value.filterBounds.return_value.filterDate.return_value.filter.return_value.sort.return_value.limit.return_value.getInfo.return_value = {\n                \'features\': candidates}\n            fetch.side_effect = [_RejectedObservation(\'No matching pixel-cloud assessment.\'),\n                                 _RejectedObservation(\'Less than 80% clear.\'),\n                                 (\'rgb\', \'valid\', \'transform\', \'crs\', {\'image_id\': \'clear\'})]\n            *_, metadata = fetch_observation(HALIFAX_BOUNDS, \'2025-06-01\', \'2025-09-01\')\n            self.assertEqual(metadata[\'image_id\'], \'clear\')\n            self.assertEqual([item[\'scene\'] for item in metadata[\'skipped_scenes\']], [\'missing-cloud\', \'too-cloudy\'])\n            self.assertEqual(fetch.call_count, 3)\n\n    def test_scene_selection_reports_exhausted_candidates(self):\n        from ship_map import fetch_observation, HALIFAX_BOUNDS, _RejectedObservation\n\n        with patch(\'ship_map.ee\') as ee, patch(\n                \'ship_map._fetch_candidate\', side_effect=_RejectedObservation(\'Area coverage below 98%.\')):\n            ee.ImageCollection.return_value.filterBounds.return_value.filterDate.return_value.filter.return_value.sort.return_value.limit.return_value.getInfo.return_value = {\n                \'features\': [{\'id\': \'partial\'}]}\n            with self.assertRaisesRegex(ValueError, r\'No usable scene among 1 candidates.*Area coverage\'):\n                fetch_observation(HALIFAX_BOUNDS, \'2025-06-01\', \'2025-09-01\')\n\n    def test_review_gate(self):\n        from canadian_validation import checked_annotation\n        sample={\'imageWidth\':128,\'imageHeight\':128,\'annotation_status\':\'UNANNOTATED\',\n                \'flags\':{\'review_required\':True},\'shapes\':[]}\n        self.assertFalse(checked_annotation(sample))\n        sample.update(annotation_status=\'reviewer_checked\',reviewer=\'test reviewer\',reviewed_utc=\'2026-11-09T00:00:00Z\',\n                      flags={\'review_required\':False})\n        self.assertFalse(checked_annotation(sample))\n        sample[\'negative_confirmed\']=True\n        self.assertTrue(checked_annotation(sample))\n        sample[\'annotation_status\']=\'draft_unreviewed\'\n        self.assertFalse(checked_annotation(sample))\n\n    def test_request_rejection_before_download(self):\n        from dashboard_adapter import DetectionRequest,run_detection\n        with patch(\'ship_map.fetch_observation\') as fetch:\n            with self.assertRaises(ValueError):\n                run_detection(\'.\',None,None,DetectionRequest((-140,42,-53,83),\'2025-06-01\',\'2025-09-01\'))\n            with self.assertRaises(ValueError):\n                DetectionRequest((-63.58,44.615,-63.545,44.64),\'2025-09-01\',\'2025-06-01\').validate()\n            with self.assertRaises(ValueError):\n                DetectionRequest((-63.58,44.615,-63.545,44.64),\'2025-06-01\',\'2025-09-01\',float(\'nan\')).validate()\n            fetch.assert_not_called()\n\n    def test_reference_filter(self):\n        from national_data import filter_ports\n        rows=[dict(name=\'Test Harbour\',province=\'ON\',subtype=\'Port\'),dict(name=\'Other\',province=\'BC\',subtype=\'Marina\')]\n        self.assertEqual(filter_ports(rows,\'ON\',\'Port\',\'HARBOUR\'),rows[:1])\n        self.assertEqual(filter_ports(rows,\'ON\',\'Marina\'),[])\n\n    def test_geographic_plan_keeps_halifax_together(self):\n        from canadian_validation import CASE_PLAN\n        groups={}\n        for _,_,group,split,_ in CASE_PLAN:\n            groups.setdefault(group,set()).add(split)\n        self.assertTrue(all(len(s)==1 for s in groups.values()))\n\n    def test_widget_filters_and_stale_result(self):\n        from ship_map import build_detection_app\n        with tempfile.TemporaryDirectory() as directory:\n            root=Path(directory)\n            (root/\'data\').mkdir()\n            (root/\'results\').mkdir()\n            (root/\'data/canada_ports_reference.csv\').write_text(\n                \'reference_id,name,province,subtype,longitude,latitude\\n1,Test Harbour,ON,Port,-79.86,43.27\\n2,Other,BC,Marina,-123.1,49.3\\n\')\n            panel,controls=build_detection_app(root,None,None,{})\n            self.assertEqual(controls[\'map\'].zoom,3)\n            self.assertEqual(len(controls[\'national_layer\'].data[\'features\']),2)\n            controls[\'province\'].value=\'ON\'\n            self.assertEqual(len(controls[\'national_layer\'].data[\'features\']),1)\n            controls[\'preset\'].value=\'Iqaluit (Arctic)\'\n            self.assertAlmostEqual(controls[\'fields\'][1].value,63.725)\n            controls[\'state\'][\'last\']=(\'stale\',\'stale\')\n            controls[\'threshold\'].value=.65\n            self.assertIsNone(controls[\'state\'][\'last\'])\n            self.assertTrue(controls[\'download_button\'].disabled)\n            controls[\'ports\'].value=False\n            self.assertEqual(controls[\'national_layer\'].data[\'features\'],[])\n\n\ndef run_checks(project):\n    import io\n    stream=io.StringIO()\n    result=unittest.TextTestRunner(stream=stream,verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(CanadaDashboardChecks))\n    report={\'execution_environment\':\'Google Colab\',\'tests\':result.testsRun,\'failures\':len(result.failures),\n            \'errors\':len(result.errors),\'passed\':result.wasSuccessful(),\'log\':stream.getvalue(),\n            \'scope\':\'Synthetic data, real widget construction and callbacks; no live EE inference or accuracy measurement.\'}\n    (Path(project)/\'results/canada_dashboard_checks.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(stream.getvalue())\n    if not result.wasSuccessful():\n        raise AssertionError(\'Dashboard regression checks failed.\')\n    return report\n', encoding='utf-8')


## National reference layer
Statistics Canada ODI v2 ports and marinas: selected infrastructure across Canada, Open Government Licence - Canada. Source collection October 2023–June 2024. Coverage varies; this is not an exhaustive or current operational inventory. Provider metadata stays under ignored data/raw/. National records are not ship annotations or detected ports. [Source](https://www150.statcan.gc.ca/n1/pub/34-26-0003/342600032023001-eng.htm).

In [ ]:
from national_data import acquire_ports
inventory = PROJECT_DIR/'data/canada_ports_reference.csv'
if inventory.is_file() and (PROJECT_DIR/'data/canada_ports_source.json').is_file():
    port_source = json.loads((PROJECT_DIR/'data/canada_ports_source.json').read_text())
else:
    _, port_source = acquire_ports(PROJECT_DIR)
print('ODI reference records:', port_source['records'])
print('Jurisdictions without ODI records:', port_source['provinces_territories_without_records'])

(PROJECT_DIR/'data/canada_facility_supplements.csv').write_text('reference_id,name,province,subtype,longitude,latitude,source_url,provider,record_type,validation_status\nYT-simpson-point-dock,Simpson Point park boat dock (site location),YT,park boat dock,-138.916,69.569,https://yukon.ca/en/outdoor-recreation-and-wildlife/camping/site/herschel-island-qikiqtaruk-territorial-park-campground,Government of Yukon,supplementary facility reference,Published campground coordinates approximate the dock site\n',encoding='utf-8')


## Regression checks
Synthetic filtering, region selection, stale-result clearing, annotation eligibility and request rejection; these checks do not establish model accuracy.

In [ ]:
from test_canada_dashboard import run_checks
checks=run_checks(PROJECT_DIR)


In [ ]:
from canadian_validation import build_review_manifest
review_rows,readiness=build_review_manifest(PROJECT_DIR)
print(json.dumps(readiness,indent=2))


## Dashboard
Choose Halifax or Vancouver to rehearse the demo. Filter the facility references, use a regional preset, draw a small rectangle or enter bounds. Select dates and scene cloud limit, then click **Load imagery** or **Detect ships**. Click a prediction to inspect its confidence and choose an export.

**Checkpoint.** If the course clone has the checkpoint at `results/models/vessel_detector.pt`, the dashboard loads it automatically. Otherwise use Colab's Files pane to upload `vessel_detector.pt` or `ships_ports_phase2_unpublished.zip` to session storage, then click **Restore checkpoint / project ZIP**. The ZIP can also restore archived runs and review images. No retraining is needed.

**Earth Engine.** Click **Authorize Earth Engine** and use a project you can access. Change `EE_PROJECT` below if needed. Requests are bounded to 16 km² and 6 km per side. The end date is exclusive.


In [ ]:
import hashlib
import shutil
from tempfile import NamedTemporaryFile
import requests
from zipfile import ZipFile
import ipywidgets as widgets
from dashboard_adapter import create_project_view
from ship_backend import load_detector

EE_PROJECT = 'fluent-edition-509318-a6'
CHECKPOINT_NAME = 'vessel_detector.pt'
CHECKPOINT_HASH = 'aa57b80886d48b592209a09e65cbd6b1b657585add4d7dd11f76867268201721'
PUBLISHED_ASSET_ROOT = 'https://raw.githubusercontent.com/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/e4ee20fbe0bf1252ed74e68037ac4c7955d5aaf3/students/oritse-tsegbemi_eyito/'
PUBLISHED_ASSETS = {
    'results/models/vessel_detector.pt': (42198971, 'aa57b80886d48b592209a09e65cbd6b1b657585add4d7dd11f76867268201721'),
    'results/ships_2025-07-26_ccc66456ac88.geojson': (6371, 'e69c0327a3e4ea5b67d85a9e11e49713d5399d95eaaac919f676c2e184b404ef'),
    'results/ships_2025-07-26_ccc66456ac88.json': (1433, '3f72a7f1d5b8b137263a03c029394e0f67a4a9c163c4ee5627fe7df43e44705a'),
    'results/ships_2025-07-26_ccc66456ac88.csv': (1035, '11938395f0a5231aaa0f7925e798fd8263d72920c59b70eb7d100ab5e7b34db3'),
    'results/ships_2025-07-26_ccc66456ac88.png': (162442, 'ef7f98a29624613c8b60102a0a5ca3d2f1c6896728d902011bdd511379a8b43c'),
    'results/ships_2025-08-12_f33c2c479429.geojson': (21174, 'e06178546a6cac92cd6f3625ee8a9fdf8b1039b15f5e79ff82c0fbd85681df5a'),
    'results/ships_2025-08-12_f33c2c479429.json': (1432, '136231e68375959d44facee2c2e4e4cae7514fe50c3c824b6ad6e97027b55750'),
    'results/ships_2025-08-12_f33c2c479429.csv': (3300, '1c403b6aae95efb6ed0da6907f3d09055b7bf44c26c361982084fa5f8241696b'),
    'results/ships_2025-08-12_f33c2c479429.png': (107587, '1a5c76c855a4cdfcc662995b910fa75a8ac1b71b163da07893e9a86b058dc687'),
    'data/canadian_pilot/english_bay_pilot.png': (220128, '1db4566d665db38a9f05a779fc965a2d2b8dd86d9111d7bed8730cd5e851f7c7'),
    'data/canadian_pilot/english_bay_pilot_mask.png': (899, '7e4b77f7cb2846faec0fae226c1fca5eecc5c73f3db646be028e217eeb15aff7'),
    'data/canadian_pilot/english_bay_pilot_scene.json': (1391, 'b486f33bfc4811c94d140e922570e20c7354aca5569917129f57ec2daf58242f'),
    'data/canadian_pilot/english_bay_pilot.json': (2030, 'a35111234e701890f26166a6e2687f5587ae3ac738b91f185b3c0b55ab9b1026'),
    'results/english_bay_pilot_metrics.json': (3218, 'ce9e339f3d1f9e482fb57d8d323ebdf8cc60a352b73e63b952f022a36ebdfe6d'),
}
checkpoint = PROJECT_DIR/'results/models'/CHECKPOINT_NAME
model = device = None
metrics = {}
view, messages = widgets.Output(), widgets.Output()
load_demo = widgets.Button(description='Load published demo files', layout=widgets.Layout(width='220px'))
restore = widgets.Button(description='Restore checkpoint / project ZIP', layout=widgets.Layout(width='245px'))
authorize = widgets.Button(description='Authorize Earth Engine', layout=widgets.Layout(width='190px'))
rebuild = widgets.Button(description='Open Ships and Ports', layout=widgets.Layout(width='175px'))
review = widgets.Button(description='Review Canadian labels', layout=widgets.Layout(width='190px'))

def verified_checkpoint(path):
    if hashlib.sha256(path.read_bytes()).hexdigest() != CHECKPOINT_HASH:
        raise ValueError('Checkpoint differs from the preserved project model.')

def restore_files(_=None):
    global model, device, metrics
    with messages:
        messages.clear_output()
        try:
            uploaded = Path('/content')/CHECKPOINT_NAME
            bundle = Path('/content/ships_ports_phase2_unpublished.zip')
            if not checkpoint.exists() and uploaded.exists():
                verified_checkpoint(uploaded)
                checkpoint.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(uploaded, checkpoint)
            if bundle.exists():
                prefix = 'students/oritse-tsegbemi_eyito/'
                with ZipFile(bundle) as archive:
                    for item in archive.infolist():
                        if item.is_dir() or not item.filename.startswith(prefix):
                            continue
                        relative = Path(item.filename[len(prefix):])
                        if (len(relative.parts) < 2 or relative.parts[0] not in {'data','results'}
                                or relative.suffix not in {'.json','.geojson','.csv','.png','.pt'}
                                or item.file_size >= 50_000_000):
                            continue
                        target = (PROJECT_DIR/relative).resolve()
                        if not target.is_relative_to(PROJECT_DIR.resolve()):
                            raise ValueError('Unsafe archive path.')
                        if target.exists():
                            continue
                        payload = archive.read(item)
                        if target.suffix == '.pt':
                            if target != checkpoint.resolve():
                                continue
                            if hashlib.sha256(payload).hexdigest() != CHECKPOINT_HASH:
                                raise ValueError('ZIP contains a different model checkpoint.')
                        target.parent.mkdir(parents=True, exist_ok=True)
                        target.write_bytes(payload)
            if not checkpoint.is_file():
                print('Upload vessel_detector.pt or the project ZIP to Colab session storage.')
                return
            verified_checkpoint(checkpoint)
            model, device, metrics = load_detector(checkpoint)
            build_review_manifest(PROJECT_DIR)
            print('Verified checkpoint loaded on', device)
            rebuild_view()
        except (OSError, ValueError, RuntimeError) as error:
            print(str(error))

def load_published_demo(_=None):
    """Restore published demo assets without replacing work already in the session."""
    load_demo.disabled = True
    try:
        with messages:
            messages.clear_output()
            print('Loading verified demo files from the course GitHub repository.')
            for relative, (expected_size, expected_hash) in PUBLISHED_ASSETS.items():
                target = PROJECT_DIR / relative
                if target.exists():
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = None
                try:
                    with requests.get(PUBLISHED_ASSET_ROOT + relative, stream=True, timeout=(10, 45)) as response:
                        response.raise_for_status()
                        digest = hashlib.sha256()
                        received = 0
                        with NamedTemporaryFile(dir=target.parent, delete=False) as handle:
                            temporary = Path(handle.name)
                            for chunk in response.iter_content(1024 * 1024):
                                received += len(chunk)
                                if received > expected_size:
                                    raise ValueError(f'Unexpected size for {relative}.')
                                digest.update(chunk)
                                handle.write(chunk)
                        if received != expected_size or digest.hexdigest() != expected_hash:
                            raise ValueError(f'Could not verify {relative}.')
                        temporary.replace(target)
                finally:
                    if temporary is not None:
                        temporary.unlink(missing_ok=True)
        restore_files()
    except (OSError, ValueError, requests.RequestException) as error:
        with messages:
            print(f'Could not load demo files: {error}')
    finally:
        load_demo.disabled = False


def rebuild_view(_=None):
    global app, controls
    with view:
        view.clear_output(wait=True)
        app, controls = create_project_view(PROJECT_DIR, model, device, metrics)
        display(app)

def authorize_ee(_=None):
    import ee
    with messages:
        messages.clear_output()
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
        print('Earth Engine ready:', ee.Number(1).getInfo())

def review_labels(_=None):
    from review_annotations import annotation_review
    with messages:
        messages.clear_output()
        try:
            display(annotation_review(PROJECT_DIR))
        except ValueError as error:
            print(error)

load_demo.on_click(load_published_demo)
restore.on_click(restore_files)
authorize.on_click(authorize_ee)
rebuild.on_click(rebuild_view)
review.on_click(review_labels)
display(widgets.HBox([load_demo, restore, authorize, rebuild, review], layout=widgets.Layout(flex_flow='row wrap')), messages, view)
if checkpoint.exists() or (Path('/content')/CHECKPOINT_NAME).exists() or Path('/content/ships_ports_phase2_unpublished.zip').exists():
    restore_files()
if 'app' not in globals():
    rebuild_view()


## Accuracy and next experiment
The previous four-epoch hard-negative experiment was rejected by validation AP50; retain the original checkpoint. No Canadian accuracy is established. The review plan groups both Halifax seasons together, separates Vancouver validation and Montreal exploratory test, and keeps aerial imagery in a separate domain. Additional Hudson Bay, Arctic, Newfoundland and Pacific North scenes are required. Previously inspected cases cannot serve as fresh independent final tests. Freeze new acquisition IDs and geographic groups before Canadian fine-tuning. Select epochs and confidence only on validation, then unlock new test scenes once.

The review panel saves drafts separately. Only named, explicitly checked labels qualify; empty checked images need explicit no-ship confirmation. Preserve ambiguous-object notes.

Future course integration: call `create_project_view(PROJECT_DIR, model, device, metrics)` from the Ships and Ports button, replacing the entire prior project view. `DetectionRequest` and `run_detection` provide the Python backend contract. This is a proposed interface, not a completed shared-dashboard integration.

## Recorded Colab checks
The results folder contains the November 9 national overview and Halifax screenshot. See the November 23 rehearsal evidence for fresh Colab checks in Halifax and Vancouver. The Canadian pilot below compares predictions with labels drawn on the frozen image.


In [ ]:
module_path = PROJECT_DIR / 'code' / 'canadian_pilot.py'
if workspace != repo or not module_path.is_file():
    module_path.write_text('"""Freeze a Canadian scene before visually reviewing vessels in Colab."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom PIL import Image\n\nCASE = "saint_john_pilot"\nPILOT_BOUNDS = {\n    CASE: (-66.091, 45.251, -66.051, 45.279),\n    "english_bay_pilot": (-123.230, 49.273, -123.179, 49.304),\n}\nSTART, END = "2025-05-01", "2025-10-01"\nCONFIDENCE, IOU = 0.55, 0.50\n\n\ndef _digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef freeze_scene(project, case=CASE):\n    """Save the exact RGB grid, usable-pixel mask and metadata, without inference."""\n    from ship_map import fetch_observation\n    from ship_backend import write_json\n\n    folder = Path(project) / "data" / "canadian_pilot"\n    folder.mkdir(parents=True, exist_ok=True)\n    bounds = PILOT_BOUNDS[case]\n    manifest_path = folder / f"{case}_scene.json"\n    image = folder / f"{case}.png"\n    mask = folder / f"{case}_mask.png"\n    label = folder / f"{case}.json"\n    if manifest_path.exists():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if _digest(image) != manifest["image_sha256"] or _digest(mask) != manifest["mask_sha256"]:\n            raise ValueError("Frozen image or mask changed.")\n        return image, manifest\n    if label.exists():\n        raise ValueError("An existing annotation requires its original frozen scene.")\n\n    rgb, valid, _, _, metadata = fetch_observation(bounds, START, END)\n    Image.fromarray(rgb).save(image)\n    Image.fromarray(valid.astype("uint8") * 255).save(mask)\n    metadata.update(case=case, image_sha256=_digest(image), mask_sha256=_digest(mask),\n                    date_range=[START, END], confidence=CONFIDENCE, match_iou=IOU)\n    write_json(label, dict(version="5.6.0", imagePath=image.name, imageData=None,\n                           imageWidth=rgb.shape[1], imageHeight=rgb.shape[0], shapes=[],\n                           flags={"review_required": True}, annotation_status="UNANNOTATED",\n                           image_sha256=metadata["image_sha256"]))\n    write_json(manifest_path, metadata)\n    return image, metadata\n\n\ndef _iou(a, b):\n    intersect = (max(0, min(a[2], b[2]) - max(a[0], b[0])) *\n                 max(0, min(a[3], b[3]) - max(a[1], b[1])))\n    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersect\n    return intersect / union if union else 0.0\n\n\ndef score_boxes(boxes, scores, targets, iou_threshold=IOU):\n    """Match each prediction and label at most once, by descending confidence."""\n    if len(boxes) != len(scores):\n        raise ValueError("Each prediction needs a confidence score.")\n    remaining = set(range(len(targets)))\n    matched = []\n    for index in sorted(range(len(scores)), key=lambda i: (-scores[i], i)):\n        overlap, target = max(((_iou(boxes[index], targets[j]), j) for j in remaining),\n                              default=(0.0, None))\n        if target is not None and overlap >= iou_threshold:\n            remaining.remove(target)\n            matched.append({"prediction": index, "label": target, "iou": overlap})\n    tp, fp, fn = len(matched), len(boxes) - len(matched), len(remaining)\n    return dict(true_positives=tp, false_positives=fp, false_negatives=fn,\n                precision=tp / (tp + fp) if tp + fp else None,\n                recall=tp / (tp + fn) if tp + fn else None, matches=matched)\n\n\ndef evaluate_frozen_scene(project, model, device, case=CASE):\n    """Score the single-scene pilot against labels reviewed before prediction."""\n    import numpy as np\n    from canadian_validation import checked_annotation\n    from ship_map import detect_raster\n    from ship_backend import write_json\n\n    project = Path(project)\n    image, scene = freeze_scene(project, case)\n    label_path = image.with_suffix(".json")\n    label = json.loads(label_path.read_text(encoding="utf-8"))\n    if (not checked_annotation(label) or label.get("image_sha256") != scene["image_sha256"]\n            or label.get("reviewed_before_predictions") is not True):\n        raise ValueError("Review the full frozen image and confirm the labels before running inference.")\n    label_sha = _digest(label_path)\n    checkpoint = project / "results/models/vessel_detector.pt"\n    rgb = np.array(Image.open(image).convert("RGB"))\n    valid = np.array(Image.open(image.with_name(f"{case}_mask.png")).convert("L")) > 0\n    if (rgb.shape[1], rgb.shape[0]) != (label["imageWidth"], label["imageHeight"]):\n        raise ValueError("Review grid differs from the frozen image.")\n    targets = [shape["points"][0] + shape["points"][1] for shape in label["shapes"]]\n    boxes, scores = detect_raster(model, device, rgb, valid, scene["confidence"])\n    boxes, scores = boxes.tolist(), scores.tolist()\n    if _digest(label_path) != label_sha:\n        raise ValueError("Labels changed while inference was running.")\n    report = dict(scene=scene["image_id"], scope="Single-scene visual-reference pilot",\n                  image_sha256=scene["image_sha256"], labels_sha256=label_sha,\n                  checkpoint_sha256=_digest(checkpoint), score_threshold=scene["confidence"],\n                  iou_threshold=scene["match_iou"], reviewer=label["reviewer"],\n                  reviewed_utc=label["reviewed_utc"], review_notes=label.get("review_notes", ""),\n                  reference_boxes=targets,\n                  prediction_boxes=boxes, prediction_scores=scores,\n                  **score_boxes(boxes, scores, targets, scene["match_iou"]))\n    write_json(project / "results" / f"{case}_metrics.json", report)\n    return report\n', encoding='utf-8')


## Canadian visual reference pilot
Authorize Earth Engine. Select Saint John or English Bay, set `FREEZE_PILOT` to True and run the scene cell. Review the full unmarked image using original pixel coordinates and save the Labelme rectangles in the adjacent JSON. Use the same frozen grid and hash. Record the reviewer, UTC time, and `reviewed_before_predictions: true` after reviewing. Set `negative_confirmed: true` only if the complete usable image was inspected and no ships could be identified.

Load the trained checkpoint before scoring. The pilot fixes confidence at 0.55 and matches at IoU 0.50. Interpret the result only for this scene and visible vessels at 10 m resolution.


In [ ]:
FREEZE_PILOT = False
PILOT_CASE = 'saint_john_pilot'  # or 'english_bay_pilot'
if FREEZE_PILOT:
    from PIL import Image
    from canadian_pilot import freeze_scene
    pilot_image, pilot_scene = freeze_scene(PROJECT_DIR, PILOT_CASE)
    print(pilot_scene['image_id'], pilot_scene['image_sha256'])
    display(Image.open(pilot_image))


In [ ]:
SCORE_PILOT = False
if SCORE_PILOT:
    from canadian_pilot import evaluate_frozen_scene
    if model is None:
        raise RuntimeError('Restore the checkpoint first.')
    print(json.dumps(evaluate_frozen_scene(PROJECT_DIR, model, device, PILOT_CASE), indent=2))
